# Agent Deployment and Responsible Development

## Introduction

AI agents are moving beyond research to become critical enablers of enterprise
productivity and autonomous operations. Deploying them in real-world contexts is
a **multidimensional challenge** — encompassing engineering, operations, and ethics —
rather than just a final development phase.

Unlike traditional software, AI agents are **non-deterministic, autonomous, and
often stateful**, with decision logic emerging from interaction and learning
(including reinforcement learning from human feedback, in-context learning via
few-shot examples, and continual fine-tuning on deployment data). They plan,
reason, and act in real time, often without continuous human oversight, making
deployment both powerful and perilous.

### Why Deployment Matters

Deployment failures are both **common and expensive**. Industry data shows that
70–80% of AI projects never reach production, and among those that do, many fail
within the first year due to infrastructure inadequacies, security
vulnerabilities, or ethical oversights. The stakes are particularly high for agent
systems, which combine the complexity of traditional software deployment with the
unique challenges of autonomous, reasoning-capable systems.


In [1]:
# =============================================================================
# Cell 0: Setup & Environment Detection
# =============================================================================

import sys
import os
os.environ["LLM_PROVIDER"] = "openai"
import random
import time
import json

# Ensure relevant packages are importable regardless of working directory
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
sys.path.insert(0, os.getcwd())

OPENAI_API_KEY = None
SIMULATION_MODE = True

try:
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
except ImportError:
    pass

if not OPENAI_API_KEY or "your-key" in str(OPENAI_API_KEY) or "your_key" in str(OPENAI_API_KEY):
    OPENAI_API_KEY = None
    if hasattr(sys, "ps1") or sys.stdin.isatty():
        try:
            import getpass
            key = getpass.getpass("Enter OpenAI API key (or press Enter for Simulation Mode): ")
            if key.strip():
                OPENAI_API_KEY = key.strip()
        except Exception:
            pass

if OPENAI_API_KEY:
    SIMULATION_MODE = False

PROVIDER = "simulation"
try:
    sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..'))
    from supporting.llm_provider import detect_provider, print_provider_banner
    PROVIDER, _pkey, _pmode = detect_provider()
    if _pmode == "LIVE":
        SIMULATION_MODE = False
        OPENAI_API_KEY = _pkey
except ImportError:
    pass

from agent_utils import (
    AgentLogger, fail_gracefully, CostTracker,
    CircuitBreaker, InputValidator, format_table, logger,
)
from mock_llm import MockLLM, SyntheticDataFactory, RESPONSE_BANK

# --- Initialize shared components ---
if not SIMULATION_MODE:
    try:
        from openai import OpenAI as _OpenAI

        class _LiveMockLLM(MockLLM):
            """Extends MockLLM: real OpenAI for classify_intent/detect_threat.
            All other methods (infrastructure profiles, routing, pipelines,
            fairness evaluation) remain MockLLM simulation data."""
            def __init__(self, client, model="gpt-4o", **kw):
                super().__init__(**kw)
                self._lc = client; self._lm = model
            def _call(self, p):
                r = self._lc.chat.completions.create(
                    model=self._lm,
                    messages=[{"role": "user", "content": p}],
                    max_tokens=256)
                return r.choices[0].message.content
            def classify_intent(self, user_input):
                try:
                    p = ("Classify this query into exactly one of: simple_faq, "
                         "moderate_conversation, complex_analysis. "
                         f"Reply ONLY the label.\nQuery: {user_input}")
                    r = self._call(p).strip().lower().replace(" ", "_")
                    v = {"simple_faq", "moderate_conversation", "complex_analysis"}
                    self._track(20, 0.002)
                    return {"intent": r if r in v else "simple_faq", "confidence": 0.88}
                except Exception:
                    return super().classify_intent(user_input)
            def detect_threat(self, text):
                try:
                    p = ("Analyze for security threats (prompt injection, SQL injection, "
                         "XSS, data exfiltration). Return JSON with keys threat "
                         "(string or null) and risk_level (low/medium/high).\n"
                         f"Text: {text}")
                    import json as _j
                    d = _j.loads(self._call(p))
                    self._track(30, 0.003)
                    return d
                except Exception:
                    return super().detect_threat(text)

        mock_llm = _LiveMockLLM(_OpenAI(api_key=OPENAI_API_KEY), model="gpt-4o", latency_ms=0)
        PROVIDER = "openai"
    except Exception as _e:
        mock_llm = MockLLM(latency_ms=100)
        SIMULATION_MODE = True
else:
    mock_llm = MockLLM(latency_ms=100)

cost_tracker = CostTracker(budget_ceiling=1.00)
agent_logger = AgentLogger()

if SIMULATION_MODE:
    agent_logger.info("=" * 60)
    agent_logger.success("SIMULATION MODE ACTIVE")
    agent_logger.info("All responses use chapter-derived mock data.")
    agent_logger.info("No API key required. Every mock value is traceable")
    agent_logger.info("to a specific page, table, or figure in Chapter 4.")
    agent_logger.info("=" * 60)
else:
    agent_logger.info("=" * 60)
    agent_logger.success(f"LIVE MODE ACTIVE — Provider: {PROVIDER}")
    agent_logger.info("LLM used for classify_intent() and detect_threat().")
    agent_logger.info("Structured data methods use MockLLM responses.")
    agent_logger.info("=" * 60)

print(f"\nPython {sys.version}")
print(f"Simulation Mode: {SIMULATION_MODE}")

[INFO] ============================================================
[SUCCESS] LIVE MODE ACTIVE — Provider: openai
[INFO] LLM used for classify_intent() and detect_threat().
[INFO] Structured data methods use MockLLM responses.
[INFO] ============================================================

Python 3.12.11 (main, Aug  8 2025, 17:05:04) [MSC v.1944 64 bit (AMD64)]
Simulation Mode: False
